# Linear systems

Three decomposition-based solvers for Ax = b, all implemented from scratch.

| Solver | Factorization | Cost | Best for |
|---|---|---|---|
| LU with partial pivoting | A = PLU | O(n^3) | General dense A |
| QR via Householder | A = QR | O(n^3), ~2x LU | Least squares (tall A) |
| Conjugate gradient | iterative | O(k·n^2) | SPD, large sparse A |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.linear_systems import lu_solve, qr_solve, conjugate_gradient

## LU decomposition with partial pivoting

Decomposes A into A = PLU where P is a permutation matrix, L is unit lower triangular, and U is upper triangular. Solve proceeds via forward/back substitution.

Partial pivoting swaps the row with the largest absolute value in the current column into the pivot position, which controls growth in the L entries and keeps the factorization numerically stable.

In [ ]:
rng = np.random.default_rng(0)
n = 50
A = rng.standard_normal((n, n))
b = rng.standard_normal(n)

x_lu    = lu_solve(A.tolist(), b.tolist())
x_numpy = np.linalg.solve(A, b)

residual = np.linalg.norm(np.array(x_lu) - x_numpy)
print(f'||x_lu - x_numpy|| = {residual:.2e}')

## QR decomposition via Householder reflections

A Householder reflector H = I - 2vv^T/||v||^2 is chosen at each step to zero out the sub-diagonal in the current column. The product of all reflectors gives Q; the upper triangular remainder is R.

More expensive than LU (~2x flops) but numerically superior for least-squares problems where A is tall (m > n).

In [ ]:
rng = np.random.default_rng(1)
m, n = 80, 40
A = rng.standard_normal((m, n))
b = rng.standard_normal(m)

x_qr = qr_solve(A.tolist(), b.tolist())
x_ls, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

print(f'||x_qr - x_lstsq|| = {np.linalg.norm(np.array(x_qr) - x_ls):.2e}')

## Conjugate gradient

An iterative method for symmetric positive definite (SPD) systems. Builds a Krylov subspace K_k = span{r_0, Ar_0, ..., A^{k-1}r_0} and minimizes the A-norm of the error over each successive subspace.

Terminates in at most n steps in exact arithmetic. In practice, converges in far fewer iterations when the eigenvalues of A are clustered.

Each iteration costs one matrix-vector product, so it's efficient for large sparse SPD systems.

In [ ]:
rng = np.random.default_rng(2)
n = 100
Q, _ = np.linalg.qr(rng.standard_normal((n, n)))
# SPD matrix with eigenvalues in [1, 10]
eigenvalues = rng.uniform(1, 10, n)
A = (Q * eigenvalues) @ Q.T
b = rng.standard_normal(n)

x_cg, residuals = conjugate_gradient(A.tolist(), b.tolist(), tol=1e-10)
x_ref = np.linalg.solve(A, b)

print(f'iterations: {len(residuals)}')
print(f'||x_cg - x_ref|| = {np.linalg.norm(np.array(x_cg) - x_ref):.2e}')

plt.semilogy(residuals)
plt.xlabel('iteration')
plt.ylabel('||r||')
plt.title('CG residual on 100x100 SPD system')
plt.tight_layout()
plt.show()

## Error analysis: LU vs numpy for ill-conditioned matrices

As the condition number grows, the residual of both solvers increases. Partial pivoting keeps the LU residual close to the numpy reference.

In [ ]:
rng = np.random.default_rng(3)
n = 10
cond_numbers = np.logspace(1, 12, 20)
lu_errors = []

for kappa in cond_numbers:
    # Build matrix with prescribed condition number
    Q1, _ = np.linalg.qr(rng.standard_normal((n, n)))
    Q2, _ = np.linalg.qr(rng.standard_normal((n, n)))
    sigma = np.logspace(0, np.log10(kappa), n)
    A = (Q1 * sigma) @ Q2.T
    b = rng.standard_normal(n)
    try:
        x_lu  = np.array(lu_solve(A.tolist(), b.tolist()))
        x_ref = np.linalg.solve(A, b)
        lu_errors.append(np.linalg.norm(x_lu - x_ref) / np.linalg.norm(x_ref))
    except Exception:
        lu_errors.append(float('nan'))

plt.loglog(cond_numbers, lu_errors, marker='o', markersize=4)
plt.xlabel('condition number')
plt.ylabel('relative error')
plt.title('LU relative error vs condition number')
plt.tight_layout()
plt.show()